In [286]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [287]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install torch # Using version 2.10.0+cu128
# %pip install torchinfo

In [288]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)
# DEVICE = "cpu"

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss
            all_preds_class = [] # Initialise list to store output prediction classes
            all_labels = [] # Initialise list to store actual labels of output predictions

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN VALIDATION)

                    # Compute confusion matrix values
                    preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
                    all_preds_class.append(preds_class.cpu())
                    all_labels.append(labels.cpu())

                    # Compute validation loss
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

                all_preds_class = torch.cat(all_preds_class)
                all_labels = torch.cat(all_labels)

                cm = confusion_matrix(all_labels, all_preds_class)
                print("Validation confusion matrix:\n", cm)

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")


##### Model evaluation function #####
def eval(
        model: nn.Module,
        test_loader: DataLoader,
        criterion: nn.Module,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model.eval() # Set model to evaluation mode
    test_loss = 0.0 # Initialise test loss
    all_preds_class = [] # Initialise list to store output prediction classes
    all_labels = [] # Initialise list to store actual labels of output predictions

    with torch.no_grad(): # Disable gradient computing
        # Loop through all batches in the test set
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN TESTING)

            # Compute confusion matrix values
            preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
            all_preds_class.append(preds_class.cpu())
            all_labels.append(labels.cpu())
            
            # Compute validation loss
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

            test_loss += loss.item() * inputs.size(0) # Sum test loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        test_loss /= len(test_loader.dataset)

        all_preds_class = torch.cat(all_preds_class)
        all_labels = torch.cat(all_labels)

        cm = confusion_matrix(all_labels, all_preds_class)
        print("Testing confusion matrix:\n", cm)

    if print_loss:
        print(f"Test Loss: {test_loss:.5f}")

2.10.0+cu128


In [289]:
### Import data ###
data1 = pd.read_csv("Data/adi_focused.csv")
data2 = pd.read_csv("Data/adi_stressed.csv")
data3 = pd.read_csv("Data/louis_focused.csv")
data4 = pd.read_csv("Data/louis_stressed.csv")

# Remove first two columns (they are just host_time and time)
data1 = data1.iloc[:, 2:]
data2 = data2.iloc[:, 2:]
data3 = data3.iloc[:, 2:]
data4 = data4.iloc[:, 2:]

In [290]:
##### Process data #####
### Function for normalising data ###
def normalise_data(data_in):
    for i in range(data_in.shape[1]):
        mean1 = np.mean(data_in[:data1.shape[0], i])
        std1 = np.std(data_in[:data1.shape[0], i])

        mean2 = np.mean(data_in[(data2.shape[0]+1):, i])
        std2 = np.std(data_in[(data2.shape[0]+1):, i])

        data_in[:data1.shape[0], i] = (data_in[:data1.shape[0], i] - mean1) / std1
        data_in[(data2.shape[0]+1):, i] = (data_in[(data2.shape[0]+1):, i] - mean2) / std2
    
    return data_in


### Form training and validation sets
split = 0.7
train_end_idx = round(data1.shape[0]*split)
data1_train = data1.iloc[0:train_end_idx]
data1_val = data1.iloc[train_end_idx+1:]

train_end_idx = round(data2.shape[0]*split)
data2_train = data2.iloc[0:train_end_idx]
data2_val = data2.iloc[train_end_idx+1:]


### Form data ###
# Vertically concatenate both data files
train_data_raw = pd.concat([data1_train, data2_train], axis=0, ignore_index=True)
val_data_raw = pd.concat([data1_val, data2_val], axis=0, ignore_index=True)
test_data_raw = pd.concat([data3, data4], axis=0, ignore_index=True)

# Convert pandas data frame into a numpy array of float32 (data should be with type float32)
train_data_raw_np = train_data_raw.values.astype("float32")
val_data_raw_np = val_data_raw.values.astype("float32")
test_data_raw_np = test_data_raw.values.astype("float32")

# Normalise data
train_data_np = normalise_data(train_data_raw_np)
val_data_np = normalise_data(val_data_raw_np)
test_data_np = normalise_data(test_data_raw_np)


### Create data LABELS ###
labels1_train = np.zeros([data1_train.shape[0], 1], dtype=np.int64) # Create a array of 0 labels for the focused dataset
labels1_val = np.zeros([data1_val.shape[0], 1], dtype=np.int64) # Create a array of 0 labels for the focused dataset
labels2_train = np.ones([data2_train.shape[0], 1], dtype=np.int64) # Create an array of 1 labels for the stressed dataset
labels2_val = np.ones([data2_val.shape[0], 1], dtype=np.int64) # Create an array of 1 labels for the stressed dataset
labels3 = np.zeros([data3.shape[0], 1], dtype=np.int64) # Create a array of 0 labels for the focused dataset
labels4 = np.ones([data4.shape[0], 1], dtype=np.int64) # Create an array of 1 labels for the stressed dataset

# Vertically concatenate both labels data files
train_labels_np = np.vstack((labels1_train, labels2_train))
val_labels_np = np.vstack((labels1_val, labels2_val))
test_labels_np = np.vstack((labels3, labels4))

# Create training, validation and test datasets
train_data = train_data_np
train_labels = train_labels_np
val_data = val_data_np
val_labels = val_labels_np
test_data = test_data_np
test_labels = test_labels_np


### DEBUGGING
# print(train_data_raw.head()) # Print first 5 rows of data for inspection
# print(test_data_raw.head()) # Print first 5 rows of data for inspection
# print(test_data.shape)
# plt.plot(train_data_np[:,2])

/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


In [291]:
##### Load and create datasets #####
# Function for obtaining windowed input-label pairs (needed as our data is highly dependent on previous data)
# E.g. [x0, x1, x2] -> y       [x1, x2, x3] -> y ...
def window_data(dataset, labels, win_size=10):
    # print(range(len(dataset) - win_size))
    # print(dataset)
    input_seq = np.array([dataset[i:i+win_size, :] for i in range(len(dataset) - win_size)]) # Get input sequence with length = window length
    seq_label = [labels[i+win_size, 0] for i in range(len(dataset) - win_size)] # Get corresponding output for each input window sequence

    # print(input_seq)
    return np.array(input_seq), np.array(seq_label)


# Initialisations/definitions
num_sensor_readings = 20 # Number of sensor readings
num_classes = 2 # Number of classification classes (number of emotional states to identify)

# Get windowed data
window_size = 16
train_inputs, train_labels = window_data(train_data, train_labels, window_size)
val_inputs, val_labels = window_data(val_data, val_labels, window_size)
test_inputs, test_labels = window_data(test_data, test_labels, window_size)

# Convert data to tensors
train_inputs_tensor = torch.tensor(train_inputs, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)#.unsqueeze(1) # nn.CrossEntropyLoss() expects integer class labels, no floats or one-hot. Also no need for unsqueeze(1) for CrossEntropyLoss, it just take labels with dim [batch size]

val_inputs_tensor = torch.tensor(val_inputs, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)#.unsqueeze(1)

test_inputs_tensor = torch.tensor(test_inputs, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)#.unsqueeze(1)

# Build data loaders
train_dataset = TensorDataset(train_inputs_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)#True)

val_dataset = TensorDataset(val_inputs_tensor, val_labels_tensor)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)#True)

test_dataset = TensorDataset(test_inputs_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)#True)


## DEBUGGING
print(f"Shape of training inputs: {train_inputs_tensor.shape}")
print(f"Shape of training labels: {train_labels_tensor.shape}")

print(f"Shape of validation inputs: {val_inputs_tensor.shape}")
print(f"Shape of validation labels: {val_labels_tensor.shape}")

print(f"Shape of testing inputs: {test_inputs_tensor.shape}")
print(f"Shape of testing labels: {test_labels_tensor.shape}")

Shape of training inputs: torch.Size([663, 16, 20])
Shape of training labels: torch.Size([663])
Shape of validation inputs: torch.Size([273, 16, 20])
Shape of validation labels: torch.Size([273])
Shape of testing inputs: torch.Size([1522, 16, 20])
Shape of testing labels: torch.Size([1522])


In [292]:
print(f"Train labels min: {train_labels_tensor.min()}, max: {train_labels_tensor.max()}")
print(f"Train labels shape: {train_labels_tensor.shape}")

Train labels min: 0, max: 1
Train labels shape: torch.Size([663])


In [293]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=32, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.relu1 = nn.ReLU() # ReLU
        self.dropout1 = nn.Dropout(0.3) # Dropout

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.relu2 = nn.ReLU() # ReLU
        self.dropout2 = nn.Dropout(0.3) # Dropout

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.3) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
        self.layernorm2 = nn.LayerNorm(output_size) # Layer norm
        self.relu4 = nn.ReLU() # ReLU
        self.dropout4 = nn.Dropout(0.3) # Dropout
    
    def forward(self, x):
        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        # out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        out = self.relu1(out)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        out = self.relu2(out)
        out = self.dropout2(out)
        out = out[:, -1, :]
        
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm2(out)
        out = self.relu4(out)
        out_logits = self.dropout4(out)

        return out_logits

In [294]:
##### Traing model #####
model = LSTMClassifier()#.to(DEVICE) # Define model
print(torchinfo.summary(model, input_size=(1, window_size, num_sensor_readings))) # Input: [batch size, sequence length, input size (number of sensors)]

# Train model
train(
    model,
    train_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    optim.Adam(model.parameters(), lr=0.001),
    num_epochs=50,#500
    val_loader=val_loader,
    print_loss=True,
)


# Test model
eval(
    model,
    test_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    print_loss=True,
)

/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Layer (type:depth-idx)                   Output Shape              Param #
LSTMClassifier                           [1, 2]                    --
├─LSTM: 1-1                              [1, 16, 32]               6,912
├─ReLU: 1-2                              [1, 16, 32]               --
├─Dropout: 1-3                           [1, 16, 32]               --
├─LSTM: 1-4                              [1, 16, 32]               8,448
├─ReLU: 1-5                              [1, 16, 32]               --
├─Dropout: 1-6                           [1, 16, 32]               --
├─Linear: 1-7                            [1, 32]                   1,056
├─LayerNorm: 1-8                         [1, 32]                   64
├─ReLU: 1-9                              [1, 32]                   --
├─Dropout: 1-10                          [1, 32]                   --
├─Linear: 1-11                           [1, 2]                    66
├─LayerNorm: 1-12                        [1, 2]                    4
├─ReLU: